# Binary CNN Promoter Classification

This notebook runs the binary BayesSigma experiment using the shared pipeline in `scripts/run_binary_experiment.py`.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_binary_experiment import (
    BinaryExperimentConfig,
    create_loaders,
    ensure_results_dirs,
    evaluate_test_set,
    load_and_split_data,
    run_conformal,
    run_mc_dropout,
    run_temperature_scaling,
    train_binary_cnn,
)
from src.evaluate import plot_training_history
from src.train import set_seed

## 2. Configuration

In [ ]:
config = BinaryExperimentConfig(
    seed=42,
    max_len=81,
    batch_size=64,
    epochs=30,
    learning_rate=1e-3,
    patience=8,
    dropout=0.3,
    mc_dropout_passes=30,
    data_dir=PROJECT_ROOT / "data",
    results_dir=PROJECT_ROOT / "results",
    class_names=("Non-Promoter", "Promoter"),
)

set_seed(config.seed)
model_dir, figure_dir, table_dir = ensure_results_dirs(config)

print(config)
print("Model directory:", model_dir)
print("Figure directory:", figure_dir)
print("Table directory:", table_dir)

## 3. Load Dataset

In [ ]:
train_split, val_split, calibration_split, test_df = load_and_split_data(config)

## 4. Split Dataset

In [ ]:
print("Train split:", train_split.shape)
print("Validation split:", val_split.shape)
print("Calibration split:", calibration_split.shape)
print("Untouched test:", test_df.shape)

## 5. Create Dataloaders

In [ ]:
train_loader, val_loader, calibration_loader, test_loader = create_loaders(
    config,
    train_split,
    val_split,
    calibration_split,
    test_df,
)

## 6. Train CNN

In [ ]:
model, history_df, device = train_binary_cnn(
    config,
    train_loader,
    val_loader,
    train_split,
    model_dir / "binary_cnn.pt",
    table_dir / "binary_training_history.csv",
)

plot_training_history(history_df, save_path=figure_dir / "binary_training_history.png")
history_df.tail()

## 7. Evaluate Classification

In [ ]:
test_labels, test_proba, test_pred, test_metrics = evaluate_test_set(
    config,
    model,
    test_loader,
    figure_dir,
    table_dir,
    device,
)

test_metrics

## 8. MC Dropout Uncertainty

In [ ]:
mc_result, uncertainty_df = run_mc_dropout(
    config,
    model,
    test_loader,
    figure_dir,
    table_dir,
    device,
)

uncertainty_df.head()

## 9. Temperature Scaling Calibration

In [ ]:
calibration_outputs = run_temperature_scaling(
    config,
    model,
    val_loader,
    calibration_loader,
    test_loader,
    test_labels,
    figure_dir,
    table_dir,
    device,
)

calibration_outputs["calibration_result"]

## 10. Conformal Prediction

In [ ]:
conformal_outputs = run_conformal(
    config,
    calibration_outputs["calibration_labels"],
    calibration_outputs["calibrated_calibration_proba"],
    test_labels,
    calibration_outputs["calibrated_test_proba"],
    table_dir,
)

conformal_outputs

## 11. Final Summary

In [ ]:
print("Binary experiment completed successfully.")
print("Best model:", model_dir / "binary_cnn.pt")
print("Training history:", table_dir / "binary_training_history.csv")
print("Metrics:", table_dir / "binary_metrics.csv")
print("Calibration metrics:", table_dir / "binary_calibration_metrics.csv")
print("Conformal metrics:", table_dir / "binary_conformal_metrics.csv")